# EDA — Paso 3: Enriquecimiento con `previous_application.csv`

**Objetivo:** Traer información de los préstamos previos internos de Home Credit hacia `application_train`.

En este Paso 3 vamos a:
1. Explorar `previous_application.csv` (solicitudes anteriores en Home Credit)
2. Analizar estados de contrato, tasas de aprobación/rechazo
3. Crear agregaciones por cliente (`SK_ID_CURR`)
4. Unir con `application_train` y analizar impacto en la TARGET

**Pregunta de negocio:** Un cliente que fue rechazado antes o que pidió montos excesivos es más riesgoso.

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Setup listo.')

---
## 2. Carga de datos

Cargamos ambas tablas. `previous_application.csv` tiene UNA fila por solicitud previa del cliente en Home Credit.

**Pregunta de negocio:** Un cliente con muchas solicitudes rechazadas es más riesgoso.

In [ ]:
app_train = pd.read_csv('../data/raw/application_train.csv')
prev = pd.read_csv('../data/raw/previous_application.csv')

print('=== application_train ===')
print(f'Filas:    {app_train.shape[0]:,}')
print(f'Columnas: {app_train.shape[1]}')
print(f'IDs:      {app_train["SK_ID_CURR"].nunique():,}')
print()
print('=== previous_application ===')
print(f'Filas:    {prev.shape[0]:,}')
print(f'Columnas: {prev.shape[1]}')
print(f'IDs:      {prev["SK_ID_CURR"].nunique():,}')

In [ ]:
# Cuántas solicitudes previas por cliente
prev_per_client = prev.groupby('SK_ID_CURR').size()

print('Solicitudes previas por cliente:')
print(prev_per_client.describe())
print(f'\nClientes sin solicitud previa: {app_train["SK_ID_CURR"].nunique() - prev["SK_ID_CURR"].nunique():,}')
print(f'Clientes con solicitud previa: {prev["SK_ID_CURR"].nunique():,}')

In [ ]:
prev.head()

---
## 3. Estructura y calidad

In [ ]:
prev.info()

In [ ]:
prev.describe()

In [ ]:
# Nulos
nulls = prev.isnull().sum()
nulls_pct = (nulls / len(prev) * 100).round(2)

null_df = pd.DataFrame({'nulos': nulls, 'pct': nulls_pct}).query('nulos > 0').sort_values('pct', ascending=False)
print(f'Columnas con nulos: {len(null_df)} de {prev.shape[1]}')
null_df

---
## 4. Análisis de estados de contrato

La variable más importante de esta tabla: `NAME_CONTRACT_STATUS`.

Los estados son:
- **Approved** → Aprobado
- **Canceled** → Cancelado por el cliente
- **Refused** → Rechazado por Home Credit
- **Unused offer** → Oferta no usada

In [ ]:
# Distribución de estados
print('ESTADOS DE CONTRATOS PREVIOS:')
status_counts = prev['NAME_CONTRACT_STATUS'].value_counts()
status_pct = prev['NAME_CONTRACT_STATUS'].value_counts(normalize=True) * 100

for status in status_counts.index:
    print(f'  {status:20s} → {status_counts[status]:>8,} ({status_pct[status]:.1f}%)')

In [ ]:
# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Approved': '#2ecc71', 'Canceled': '#f39c12', 'Refused': '#e74c3c', 'Unused offer': '#95a5a6'}
status_order = prev['NAME_CONTRACT_STATUS'].value_counts().index
bar_colors = [colors.get(s, '#3498db') for s in status_order]

status_counts.plot(kind='bar', ax=axes[0], color=bar_colors)
axes[0].set_title('Solicitudes previas por estado')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=45)

status_pct.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=bar_colors)
axes[1].set_title('Proporción de estados')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Tipo de cliente vs estado
print('TIPO DE CLIENTE POR ESTADO:')
ct = pd.crosstab(prev['NAME_CLIENT_TYPE'], prev['NAME_CONTRACT_STATUS'], normalize='index') * 100
ct.round(1)

In [ ]:
# Monto solicitado vs aprobado
prev['AMT_DIFF'] = prev['AMT_CREDIT'] - prev['AMT_APPLICATION']
prev['AMT_DIFF_PCT'] = (prev['AMT_CREDIT'] / prev['AMT_APPLICATION'].replace(0, np.nan) - 1) * 100

print('Diferencia entre monto solicitado y aprobado:')
print(prev.groupby('NAME_CONTRACT_STATUS')[['AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DIFF']].mean())

---
## 5. Agregaciones por cliente

Comprimimos las múltiples solicitudes previas en UNA fila por cliente.

In [ ]:
# === AGREGACIONES NUMÉRICAS ===
num_agg = prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_APPLICATION': ['sum', 'mean', 'max'],
    'AMT_CREDIT': ['sum', 'mean', 'max'],
    'AMT_ANNUITY': ['mean', 'max'],
    'AMT_DOWN_PAYMENT': ['sum', 'mean'],
    'DAYS_DECISION': ['mean', 'min', 'max'],
    'CNT_PAYMENT': ['mean', 'max'],
    'RATE_DOWN_PAYMENT': 'mean',
})

num_agg.columns = ['_'.join(col).strip() for col in num_agg.columns]
num_agg = num_agg.reset_index()

# Renombrar para claridad
num_agg = num_agg.rename(columns={
    'SK_ID_PREV_count': 'prev_total_applications',
    'AMT_APPLICATION_sum': 'prev_total_requested',
    'AMT_APPLICATION_mean': 'prev_avg_requested',
    'AMT_CREDIT_sum': 'prev_total_approved',
    'AMT_CREDIT_mean': 'prev_avg_approved',
    'DAYS_DECISION_mean': 'prev_avg_decision_days',
})

print(f'Clientes con agregaciones: {num_agg.shape[0]:,}')
print(f'Features generadas: {num_agg.shape[1] - 1}')
num_agg.head()

In [ ]:
# === PROPORCIONES DE ESTADO ===
status_dummies = pd.get_dummies(prev[['SK_ID_CURR', 'NAME_CONTRACT_STATUS']])

status_agg = status_dummies.groupby('SK_ID_CURR').sum().reset_index()

total_apps = prev.groupby('SK_ID_CURR').size().reset_index(name='prev_total_applications')
status_agg = status_agg.merge(total_apps, on='SK_ID_CURR')

# Calcular proporciones
status_cols = [c for c in status_agg.columns if c.startswith('NAME_CONTRACT_STATUS_')]
for col in status_cols:
    status_agg[f'{col}_pct'] = status_agg[col] / status_agg['prev_total_applications']

print('Proporciones de estado calculadas.')

In [ ]:
# === PROPORCIONES DE TIPO DE CLIENTE ===
client_dummies = pd.get_dummies(prev[['SK_ID_CURR', 'NAME_CLIENT_TYPE']])

client_agg = client_dummies.groupby('SK_ID_CURR').sum().reset_index()
client_agg = client_agg.merge(total_apps, on='SK_ID_CURR')

client_cols = [c for c in client_agg.columns if c.startswith('NAME_CLIENT_TYPE_')]
for col in client_cols:
    client_agg[f'{col}_pct'] = client_agg[col] / client_agg['prev_total_applications']

print('Proporciones de tipo de cliente calculadas.')

In [ ]:
# === FEATURES DERIVADAS DE NEGOCIO ===
derived = prev.groupby('SK_ID_CURR').agg({
    'NAME_CONTRACT_STATUS': lambda x: (x == 'Refused').sum(),
    'NAME_CONTRACT_STATUS': lambda x: (x == 'Approved').sum(),
})
# pandas no permite duplicar keys, lo hacemos por separado

refused_count = prev.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].apply(lambda x: (x == 'Refused').sum()).reset_index(name='prev_refused_count')
approved_count = prev.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].apply(lambda x: (x == 'Approved').sum()).reset_index(name='prev_approved_count')

derived = refused_count.merge(approved_count, on='SK_ID_CURR')
derived = derived.merge(total_apps, on='SK_ID_CURR')

# Ratios de negocio
derived['prev_reject_rate'] = derived['prev_refused_count'] / derived['prev_total_applications']
derived['prev_approve_rate'] = derived['prev_approved_count'] / derived['prev_total_applications']

print('Features derivadas de negocio:')
derived.head()

---
## 6. Merge con application_train

In [ ]:
TARGET_COL = 'TARGET'
df = app_train.copy()

# Merge numérico
df = df.merge(num_agg, on='SK_ID_CURR', how='left')

# Merge proporciones de estado (sin prev_total_applications, ya viene de num_agg)
status_merge_cols = ['SK_ID_CURR'] + [c for c in status_agg.columns if c.endswith('_pct')]
df = df.merge(status_agg[status_merge_cols], on='SK_ID_CURR', how='left')

# Merge proporciones de tipo de cliente
client_merge_cols = ['SK_ID_CURR'] + [c for c in client_agg.columns if c.endswith('_pct')]
df = df.merge(client_agg[client_merge_cols], on='SK_ID_CURR', how='left')

# Merge features derivadas
df = df.merge(derived[['SK_ID_CURR', 'prev_refused_count', 'prev_approved_count', 
                        'prev_reject_rate', 'prev_approve_rate']], on='SK_ID_CURR', how='left')

print(f'Shape final: {df.shape}')
print(f'Nuevas columnas: {df.shape[1] - app_train.shape[1]}')

---
## 7. Análisis de features enriquecidas vs TARGET

In [ ]:
# Columnas nuevas
new_cols = [c for c in df.columns if c not in app_train.columns]

# Correlaciones con TARGET
corr_with_target = df[new_cols + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)

print('=== TOP 15 POSITIVAS (mayor riesgo de default) ===')
print(corr_with_target.sort_values(ascending=False).head(15))
print()
print('=== TOP 10 NEGATIVAS (menor riesgo) ===')
print(corr_with_target.sort_values(ascending=True).head(10))

In [ ]:
# Visualización: Top features correlacionadas con TARGET
top_corr = corr_with_target.abs().sort_values(ascending=False).head(15)
top_features = top_corr.index.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
corr_with_target[top_features].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 features de PREVIOUS APPLICATION correlacionadas con TARGET')
ax.set_xlabel('Correlación de Pearson')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Default rate por tasa de rechazo previo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Tasa de rechazo vs TARGET
df.groupby(TARGET_COL)['prev_reject_rate'].mean().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Tasa de Rechazo Previa vs TARGET')
axes[0].set_xticklabels(['Pagó (0)', 'Default (1)'], rotation=0)

# Tasa de aprobación vs TARGET
df.groupby(TARGET_COL)['prev_approve_rate'].mean().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Tasa de Aprobación Previa vs TARGET')
axes[1].set_xticklabels(['Pagó (0)', 'Default (1)'], rotation=0)

# Número de solicitudes vs TARGET
df.groupby(TARGET_COL)['prev_total_applications'].mean().plot(kind='bar', ax=axes[2], color=['#2ecc71', '#e74c3c'])
axes[2].set_title('Solicitudes Previas Promedio vs TARGET')
axes[2].set_xticklabels(['Pagó (0)', 'Default (1)'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de correlaciones
prev_features = [c for c in new_cols if 'prev_' in c][:12]
prev_features.append(TARGET_COL)

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df[prev_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlaciones: Features de PREVIOUS APPLICATION + TARGET')
plt.tight_layout()
plt.show()

---
## 8. Impacto en la TARGET

In [ ]:
# Clientes CON solicitud previa vs SIN
has_prev = df['prev_total_applications'].notna()

print('=== IMPACTO DE SOLICITUDES PREVIAS EN LA TARGET ===')
print()
print(f'Con solicitud previa:  {has_prev.sum():,} clientes')
print(f'  Default rate:        {df[has_prev][TARGET_COL].mean()*100:.2f}%')
print()
print(f'Sin solicitud previa:  {(~has_prev).sum():,} clientes')
print(f'  Default rate:        {df[~has_prev][TARGET_COL].mean()*100:.2f}%')

In [ ]:
# Default rate por nivel de rechazo previo
df['reject_range'] = pd.cut(df['prev_reject_rate'].fillna(-0.1), 
                            bins=[-0.1, 0, 0.3, 0.6, 1.0],
                            labels=['Sin rechazos', 'Bajo (<30%)', 'Medio (30-60%)', 'Alto (>60%)'])

print('Default rate por tasa de rechazo previo:')
reject_default = df.groupby('reject_range')[TARGET_COL].agg(['mean', 'count'])
reject_default.columns = ['default_rate', 'n_clients']
reject_default['default_rate'] = reject_default['default_rate'] * 100
reject_default

---
## 9. Guardar dataset enriquecido

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/application_train_prev.csv', index=False)
print(f'Guardado: {df.shape[0]:,} filas x {df.shape[1]} columnas')

---
## 10. Hallazgos y conclusiones

Completá esta sección con lo que encontraste.

### Estructura sugerida:

```markdown
### Datos de previous_application
- [ ] Total de solicitudes previas
- [ ] Proporción de estados (Approved, Refused, Canceled)

### Features más relevantes
- [ ] Top 3 features que más correlacionan con TARGET
- [ ] ¿La tasa de rechazo predice default?

### Impacto
- [ ] Default rate: con/sin solicitud previa
- [ ] Default rate por nivel de rechazo

### Próximos pasos
```

In [ ]:
# Tu resumen ejecutivo acá
print('EJECUTIVO DEL EDA - Paso 3 (Previous Application)')
print('=' * 50)
print()
print('1. PREVIOUS APPLICATION:')
print(f'   - {prev.shape[0]:,} solicitudes previas registradas')
print(f'   - {prev["SK_ID_CURR"].nunique():,} clientes con solicitud previa')
print()
print('2. FEATURES GENERADAS:')
print(f'   - {len(new_cols)} nuevas features agregadas')
print()
print('3. TOP CORRELACIONES CON TARGET:')
top5 = corr_with_target.abs().sort_values(ascending=False).head(5)
for feat, val in top5.items():
    direction = '+' if corr_with_target[feat] > 0 else '-'
    print(f'   - {feat}: {direction}{val:.4f}')
print()
print('4. IMPACTO:')
if has_prev.sum() > 0:
    rate_with = df[has_prev][TARGET_COL].mean()*100
    rate_without = df[~has_prev][TARGET_COL].mean()*100
    print(f'   - Default rate con solicitud previa: {rate_with:.2f}%')
    print(f'   - Default rate sin solicitud previa: {rate_without:.2f}%')
print()
print('5. RESUMEN DE EDA:')
print('   - 3 tablas exploradas: application, bureau, previous_application')
print('   - Próximo paso: Feature Engineering (04_feature_engineering.ipynb)')